# Phase 4 - LoRA Fine-Tuning
This notebook clones the repo and runs `src.finetune.train`.

**Kaggle-specific setup:**
- Toggle Internet **ON** in notebook settings before running
- Select GPU **T4 x2** as the accelerator
- Add `HF_TOKEN` under Add-ons > Secrets

In [ ]:
# Cell 1: GPU sanity check — must pass before proceeding
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Settings > Accelerator and select GPU T4 x2, "
        "then re-run all cells."
    )

print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Clone repo and install dependencies
!git clone https://github.com/SharjilSharma/FineTuningProject.git
%cd FineTuningProject
!pip install -q -r requirements.txt
!pip install -q trl peft bitsandbytes huggingface_hub

In [ ]:
# Cell 3: Load secrets — fails loud and fast if HF_TOKEN is missing
from kaggle_secrets import UserSecretsClient
import os

try:
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    if not hf_token:
        raise ValueError("HF_TOKEN secret is empty.")
    os.environ['HF_TOKEN'] = hf_token
    print("✅ HF_TOKEN loaded.")
except Exception as e:
    raise RuntimeError(
        f"HF_TOKEN not found in Kaggle Secrets: {e}\n"
        "Add it via the Add-ons > Secrets panel and re-run."
    ) from e

# Optional: set the HF repo to push the adapter to after training
HF_REPO_ID = "sharjilsharma/earnings-signal-lora-adapter"  # change if needed

In [ ]:
# Cell 4: Run fine-tuning
!python -m src.finetune.train

In [ ]:
# Cell 5: Push LoRA adapter to Hugging Face Hub (private repo)
# This is CRITICAL — Kaggle VMs are ephemeral; anything not pushed is lost on disconnect.
from huggingface_hub import HfApi
from src.finetune.config import FinetuneConfig

cfg = FinetuneConfig()
adapter_dir = cfg.output_dir  # e.g. data/models/finetuned_adapter

api = HfApi(token=os.environ['HF_TOKEN'])

# Create the repo if it doesn't exist yet (private by default)
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)

api.upload_folder(
    folder_path=adapter_dir,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Phase 4: LoRA adapter trained on bootstrap dataset",
)

print(f"✅ Adapter pushed to https://huggingface.co/{HF_REPO_ID}")